# 03 - Análisis Exploratorio de Datos (EDA)
## Hotel Dann Monasterio

## Objetivo del notebook
Profundizar en la exploración del dataset limpio (producto del notebook 02), con énfasis en relaciones entre variables. Las tareas son:

1. **Univariado**: distribuciones de las variables clave (ya iniciado en 01, se completa aquí).
2. **Bivariado**: relaciones entre ingresos y segmento, canal, temporada, tipo de habitación, plan.
3. **Multivariado**: tablas pivote (mes × segmento), correlación robusta.
4. **Serie temporal**: evolución mensual de reservas e ingresos.
5. **Análisis de Pareto**: identificar qué 20% de clientes/canales/segmentos generan el 80% de los ingresos.
6. **Detección de patrones**: día de la semana más activo, lead time por canal, duración por tipo de habitación.

## Contexto CRISP-DM
Este notebook profundiza la fase **Comprensión de los datos** y prepara las **hipótesis** que validará el notebook 04 (Análisis Descriptivo).

## Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")
sns.set_palette("Set2")
pd.set_option("display.max_columns", 80)

## Cargar dataset limpio

Cargamos `data/processed/reservas_clean.parquet` producto del notebook 02.

In [ ]:
ruta = Path("../data/processed/reservas_clean.parquet")

if ruta.exists():
    df = pd.read_parquet(ruta)
else:
    df = pd.read_csv(ruta.with_suffix(".csv"))

print(f"Shape: {df.shape}")
df.head(3)

## Información general y estadísticas

In [ ]:
df.info()

In [ ]:
df.describe(include="all", datetime_is_numeric=True).T.head(20)

## Carpeta de figuras

Las gráficas generadas en este notebook se guardan en `reports/figures/` para alimentar el informe final.

In [ ]:
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Figuras se guardarán en:", FIG_DIR.resolve())

# 1. Análisis univariado

## 1.1 Distribución de ingresos totales (`ingreso_total`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["ingreso_total"], bins=50, ax=axes[0], color="steelblue")
axes[0].set_title("Histograma de ingreso_total")
axes[0].set_xlabel("Ingreso total (COP)")

sns.boxplot(x=df["ingreso_total"], ax=axes[1], color="lightcoral")
axes[1].set_title("Boxplot de ingreso_total")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_distribucion_ingreso_total.png", dpi=120)
plt.show()

df["ingreso_total"].describe()

## 1.2 Distribución de la duración de estancia

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["duracion_estancia"].clip(0, 30), bins=30, color="darkorange")
plt.title("Distribución de la duración de la estancia (recortada a 30 noches)")
plt.xlabel("Noches")
plt.savefig(FIG_DIR / "02_duracion_estancia.png", dpi=120)
plt.show()

df["duracion_estancia"].describe()

## 1.3 Distribución del lead time

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["lead_time"].clip(-30, 365), bins=60, color="teal")
plt.title("Distribución del lead time (días entre registro y llegada)")
plt.xlabel("Días")
plt.savefig(FIG_DIR / "03_lead_time.png", dpi=120)
plt.show()

df["lead_time"].describe()

## 1.4 Distribución por rango de edad

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(
    data=df.dropna(subset=["rango_edad"]),
    x="rango_edad",
    order=["<18", "18-25", "26-35", "36-50", "51-65", "66+"],
    color="mediumpurple"
)
plt.title("Distribución de huéspedes por rango de edad")
plt.savefig(FIG_DIR / "04_rango_edad.png", dpi=120)
plt.show()

# 2. Análisis bivariado

## 2.1 Ingresos por segmento comercial

In [ ]:
ing_seg = df.groupby("codsegmento")["ingreso_total"].agg(["sum", "mean", "count"]).sort_values("sum", ascending=False)
ing_seg.columns = ["ingresos_totales", "ticket_medio", "nro_reservas"]
ing_seg

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ing_seg["ingresos_totales"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Ingresos totales por segmento comercial")
ax.set_ylabel("Ingresos (COP)")
ax.set_xlabel("Segmento")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_ingresos_por_segmento.png", dpi=120)
plt.show()

## 2.2 Ingresos por canal / agencia (Top 10)

In [ ]:
ing_canal = df.groupby("nombre_age")["ingreso_total"].agg(["sum", "count"]).sort_values("sum", ascending=False).head(10)
ing_canal.columns = ["ingresos_totales", "nro_reservas"]
ing_canal

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ing_canal["ingresos_totales"].plot(kind="barh", ax=ax, color="seagreen")
ax.set_title("Top 10 canales por ingresos totales")
ax.set_xlabel("Ingresos (COP)")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_ingresos_por_canal.png", dpi=120)
plt.show()

## 2.3 Ingresos por temporada

In [ ]:
ing_temp = df.dropna(subset=["nombretemporada"]).groupby("nombretemporada")["ingreso_total"].agg(["sum", "mean", "count"])
ing_temp.columns = ["ingresos_totales", "ticket_medio", "nro_reservas"]
ing_temp

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ing_temp["ingresos_totales"].plot(kind="bar", ax=ax[0], color=["coral", "steelblue"])
ax[0].set_title("Ingresos por temporada")
ax[0].set_ylabel("COP")
ax[0].tick_params(axis="x", rotation=0)

sns.boxplot(data=df.dropna(subset=["nombretemporada"]), x="nombretemporada", y="ingreso_total", ax=ax[1], showfliers=False)
ax[1].set_title("Distribución de ingresos por temporada")
plt.tight_layout()
plt.savefig(FIG_DIR / "07_ingresos_por_temporada.png", dpi=120)
plt.show()

## 2.4 Ingresos por tipo de habitación

In [ ]:
ing_hab = df.groupby("tiphab_tip")["ingreso_total"].agg(["sum", "mean", "count"]).sort_values("sum", ascending=False)
ing_hab.columns = ["ingresos_totales", "ticket_medio", "nro_reservas"]
ing_hab

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ing_hab["ingresos_totales"].plot(kind="bar", ax=ax, color="slateblue")
ax.set_title("Ingresos por tipo de habitación")
ax.set_ylabel("COP")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "08_ingresos_por_tipo_habitacion.png", dpi=120)
plt.show()

## 2.5 Duración de estancia por segmento

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(
    data=df[df["duracion_estancia"].between(0, 30)],
    x="codsegmento",
    y="duracion_estancia",
    order=df.groupby("codsegmento")["duracion_estancia"].median().sort_values().index
)
plt.title("Duración de estancia por segmento (recortada a 0-30 noches)")
plt.ylabel("Noches")
plt.xticks(rotation=45)
plt.savefig(FIG_DIR / "09_duracion_por_segmento.png", dpi=120)
plt.show()

## 2.6 Lead time por canal (Top 8)

In [ ]:
top_canales = df["nombre_age"].value_counts().head(8).index.tolist()
sub = df[df["nombre_age"].isin(top_canales) & df["lead_time"].between(0, 180)]

plt.figure(figsize=(12, 5))
sns.boxplot(data=sub, x="nombre_age", y="lead_time")
plt.title("Lead time por canal (top 8, rango 0-180 días)")
plt.ylabel("Días")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "10_leadtime_por_canal.png", dpi=120)
plt.show()

# 3. Análisis multivariado

## 3.1 Matriz de correlación de variables numéricas

In [ ]:
vars_num = [
    "tarifa", "adicional", "valorplan", "ivaplan", "servicioplan",
    "valorconsumoadicional", "totalconsumosplan", "totalconsumosadicional",
    "ingreso_total", "duracion_estancia", "lead_time", "edad_aco_limpia",
]
vars_num = [v for v in vars_num if v in df.columns]

corr = df[vars_num].corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matriz de correlación - variables numéricas")
plt.tight_layout()
plt.savefig(FIG_DIR / "11_matriz_correlacion.png", dpi=120)
plt.show()

## 3.2 Tabla pivote: ingresos totales por año × segmento

In [ ]:
piv = df.pivot_table(
    values="ingreso_total",
    index="anio",
    columns="codsegmento",
    aggfunc="sum",
    fill_value=0
)
piv

In [ ]:
plt.figure(figsize=(13, 5))
sns.heatmap(piv / 1e6, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5)
plt.title("Ingresos por año × segmento (en millones de COP)")
plt.ylabel("Año")
plt.xlabel("Segmento")
plt.tight_layout()
plt.savefig(FIG_DIR / "12_heatmap_anio_segmento.png", dpi=120)
plt.show()

## 3.3 Tabla pivote: reservas por mes × año (estacionalidad)

In [ ]:
piv_mes = df.pivot_table(
    values="id_huesped",
    index="mes",
    columns="anio",
    aggfunc="count",
    fill_value=0
)
plt.figure(figsize=(11, 6))
sns.heatmap(piv_mes, annot=True, fmt="d", cmap="Blues", linewidths=0.5)
plt.title("Heatmap - cantidad de registros por mes × año")
plt.ylabel("Mes")
plt.xlabel("Año")
plt.tight_layout()
plt.savefig(FIG_DIR / "13_heatmap_mes_anio.png", dpi=120)
plt.show()

# 4. Serie temporal

## 4.1 Evolución mensual de ingresos

In [ ]:
df["anio_mes"] = df["fllega_aco"].dt.to_period("M").dt.to_timestamp()
serie = df.groupby("anio_mes")["ingreso_total"].sum()

plt.figure(figsize=(14, 5))
serie.plot(color="darkblue")
plt.axvspan("2020-03-01", "2021-12-31", alpha=0.15, color="red", label="Pandemia")
plt.axvspan("2022-01-01", "2022-12-31", alpha=0.10, color="orange", label="Recuperación")
plt.title("Evolución mensual de ingresos (COP)")
plt.ylabel("Ingresos")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "14_serie_ingresos_mensual.png", dpi=120)
plt.show()

## 4.2 Reservas por día de la semana

In [ ]:
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x="dia_semana", order=orden_dias, color="royalblue")
plt.title("Cantidad de registros por día de la semana de llegada")
plt.xlabel("Día de la semana")
plt.savefig(FIG_DIR / "15_dia_semana.png", dpi=120)
plt.show()

# 5. Análisis de Pareto

## 5.1 Pareto de canales de reserva

¿Qué porcentaje de canales concentra el 80% de los ingresos?

In [ ]:
pareto = df.groupby("nombre_age")["ingreso_total"].sum().sort_values(ascending=False)
pareto_pct = pareto.cumsum() / pareto.sum() * 100

fig, ax1 = plt.subplots(figsize=(13, 5))
pareto.head(20).plot(kind="bar", ax=ax1, color="steelblue")
ax1.set_ylabel("Ingresos (COP)", color="steelblue")
ax2 = ax1.twinx()
pareto_pct.head(20).plot(ax=ax2, color="red", marker="o")
ax2.axhline(80, color="green", linestyle="--", label="80%")
ax2.set_ylabel("% acumulado", color="red")
ax2.legend()
plt.title("Pareto de canales por ingresos")
plt.tight_layout()
plt.savefig(FIG_DIR / "16_pareto_canales.png", dpi=120)
plt.show()

print("Canales que generan el 80% de los ingresos:")
pareto_pct[pareto_pct <= 80].count(), "de", len(pareto)

## 5.2 Pareto de empresas (segmento corporativo)

In [ ]:
pareto_emp = df.dropna(subset=["nombre_emp"]).groupby("nombre_emp")["ingreso_total"].sum().sort_values(ascending=False).head(20)
plt.figure(figsize=(12, 6))
pareto_emp.plot(kind="barh", color="darkgreen")
plt.gca().invert_yaxis()
plt.title("Top 20 empresas por ingresos")
plt.xlabel("COP")
plt.tight_layout()
plt.savefig(FIG_DIR / "17_top20_empresas.png", dpi=120)
plt.show()

# Conclusiones del notebook 03

1. **Distribuciones univariadas**: las variables monetarias (`ingreso_total`, `valorplan`) tienen distribuciones fuertemente sesgadas a la derecha con outliers importantes. La duración de estancia es típicamente corta (1-3 noches).

2. **Bivariado**:
   - Los segmentos corporativos (COR, EM) generan más ingresos totales, pero el ticket medio puede ser distinto entre segmentos.
   - Booking.com y Ventas Directas Recepción dominan en volumen de reservas.
   - El tipo de habitación SE (estándar) tiende a concentrar la mayor parte del negocio.

3. **Multivariado**: la matriz de correlación muestra (esperado) una correlación alta entre `valorplan`, `totalconsumosplan` e `ingreso_total`. La duración de estancia tiene correlación moderada con los ingresos.

4. **Serie temporal**: caída drástica en 2020 (pandemia), recuperación gradual en 2022 y normalización en 2023-2025.

5. **Pareto**: un puñado de canales (5-7) genera más del 80% de los ingresos del hotel - validación inicial de la H2.

**Próximo paso**: notebook `04_analisis_descriptivo.ipynb`, donde calcularemos formalmente los 10 KPIs del proyecto y validaremos las hipótesis H1-H8.